<a href="https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Subhash-2910/flyrank-ML-T1/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

I have choosen the Refresh / Content Opportunity Scoring. This is mainly a ranking/scoring task because the goal is to place content pages in priority order for human review. The output is not an automatic decision; it is a ranked queue that helps an SEO editor decide which pages to investigate first for refresh, expansion, protection, pruning, or monitoring.

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

For the starter dataset, I will use a temporary proxy target called is_declining_proxy. It equals 1 when trend_direction is "down" and 0 otherwise.

This is a defined rule, not a future observed outcome: it comes from the current snapshot’s trend fields. It is useful for practising the ML workflow, but it does not mean a declining page should definitely be refreshed. In a stronger later version, the target should be an observed outcome in a future time window.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # find the repo root from wherever this kernel started
    while not os.path.isdir("data/raw") and os.getcwd() != "/":
        os.chdir("..")

print("Working dir:", os.getcwd())
assert os.path.exists("data/raw/content_refresh_anonymized.csv"), "starter CSV not found — are you at the repo root?"
print("Starter data found. You're ready.")


Working dir: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
Starter data found. You're ready.


In [11]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,}")
print(f"Columns: {df.shape[1]}")

Rows: 30,000
Columns: 44


In [12]:
df["is_declining_proxy"] = (df["trend_direction"] == "down").astype(int)

print(df["is_declining_proxy"].value_counts())
print(f"Proxy decline rate: {df['is_declining_proxy'].mean():.1%}")

is_declining_proxy
1    16262
0    13738
Name: count, dtype: int64
Proxy decline rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My success metric will be precision@20. It answers: “Of the top 20 pages recommended for review, what percentage matches the target or proxy?”

This is appropriate because an editor has limited time and is likely to start with the first few pages in the queue. A higher precision@20 is better. The scoring method should beat a transparent baseline score when both are evaluated on the same held-out data.

In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
baseline_rate = df["is_declining_proxy"].mean()

print(f"Overall proxy decline rate: {baseline_rate:.1%}")
print("Success metric: precision@20 — higher than the baseline is better.")

Overall proxy decline rate: 54.2%
Success metric: precision@20 — higher than the baseline is better.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row represents one pseudonymized content page. Each row contains page-level signals from the trailing 90-day period, such as impressions, sessions, freshness, average position, and engagement.

In [14]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
unit_of_analysis = df[
    [
        "content_type",
        "impressions_90d",
        "sessions_90d",
        "days_since_last_update",
        "avg_position",
        "engagement_rate",
        "is_declining_proxy",
    ]
]

display(unit_of_analysis.head())
print(f"One row = one content page. Total page rows: {len(unit_of_analysis):,}")


,content_type,impressions_90d,sessions_90d,days_since_last_update,avg_position,engagement_rate,is_declining_proxy
0,keyword article,3803,17,20,10.6,5.88,1
1,keyword article,15320,9,25,20.3,0.00,1
2,keyword article,12581,11,20,36.5,0.00,1
3,keyword article,11751,78,22,6.2,1.28,0
4,keyword article,19140,145,14,44.0,0.00,1


One row = one content page. Total page rows: 30,000


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule such as “review every page older than 180 days” is easy to understand, but it ignores important context. For example, an old page with no visibility may be less urgent than a visible page with weak engagement, a page-one ranking that is declining, or a page with good demand but low CTR.

A scoring approach can combine several signals—visibility, freshness, average position, sessions, and engagement—and rank pages based on their combined pattern. I will begin with transparent baseline rules and only use a more complex model if it improves precision@20 while still giving understandable reason codes.

In [15]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
planned_features = {
    "impressions_90d",
    "sessions_90d",
    "days_since_last_update",
    "avg_position",
    "engagement_rate",
}

forbidden_leakage_features = {"trend_direction", "trend_pct"}

assert planned_features.isdisjoint(forbidden_leakage_features)
print("Leakage check passed: trend fields are not planned features.")

Leakage check passed: trend fields are not planned features.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.